<a href="https://colab.research.google.com/github/DCOMP-UFS/Engenharia_SoftwareII_2025-2_T04_screenpipe/blob/main/Modelo_1_bge_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Thu Nov 13 04:46:42 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             31W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
torch.cuda.empty_cache()

In [ ]:
# 1. Instalar as bibliotecas necessárias
!pip install transformers accelerate bitsandbytes sentence-transformers Pillow datasets
!pip install -q git+https://github.com/huggingface/peft.git  # PEFT é útil para modelos grandes

# 2. Clonar o repositório ScreenPipe
!git clone https://github.com/mediar-ai/screenpipe.git
%cd screenpipe
!ls # Verifique os diretórios clonados

In [ ]:
!pip install -q sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer

# Modelo 1: BGE-base para embeddings
embed_model_name = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(embed_model_name, device="cuda")

print("Modelo de embeddings carregado:", embed_model_name)


In [ ]:
import os

%cd /content/screenpipe

# extensões que queremos indexar
EXTENSOES_VALIDAS = (".rs", ".ts", ".tsx", ".js", ".md", ".toml", ".yaml", ".yml")

docs = []   # cada item: {"texto": ..., "arquivo": ..., "modulo": ...}

for root, dirs, files in os.walk("."):
    for fname in files:
        if fname.endswith(EXTENSOES_VALIDAS):
            path = os.path.join(root, fname)
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
            except Exception:
                continue

            # módulo = nome da primeira pasta depois de ./  (ex: ./screenpipe-core/src/... -> screenpipe-core)
            rel = path.lstrip("./")
            modulo = rel.split(os.sep)[0] if os.sep in rel else rel

            # dividir em chunks de ~1000 caracteres
            CHUNK_SIZE = 1000
            for i in range(0, len(text), CHUNK_SIZE):
                chunk = text[i:i+CHUNK_SIZE]
                if chunk.strip():
                    docs.append({
                        "texto": chunk,
                        "arquivo": path,
                        "modulo": modulo,
                        "offset": i,  # onde começa no arquivo
                    })

print(f"Total de chunks criados: {len(docs)}")
print("Exemplo de doc:", docs[0]["arquivo"], "| módulo:", docs[0]["modulo"])


In [ ]:
from tqdm.auto import tqdm

corpus_texts = [d["texto"] for d in docs]

# gera embeddings normalizados (bom pra cosine similarity)
embeddings = embed_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Shape dos embeddings:", embeddings.shape)


In [ ]:
import numpy as np
from sentence_transformers.util import cos_sim

def buscar_trechos(query, modulo=None, top_k=5):
    """
    query: texto da busca (ex: 'captura de tela', 'API de busca', 'pipes/plugins')
    modulo: se quiser filtrar por 'screenpipe-core', 'screenpipe-server', etc.
    top_k: quantos trechos retornar
    """
    # embedding da consulta
    q_emb = embed_model.encode([query], normalize_embeddings=True)[0]

    # similaridade com todos os chunks
    scores = cos_sim(q_emb, embeddings)[0].cpu().numpy()

    # se quiser filtrar por módulo:
    indices_validos = list(range(len(docs)))
    if modulo:
        indices_validos = [i for i, d in enumerate(docs) if d["modulo"] == modulo]

    # pegar só os scores desses índices
    scores_filtrados = np.array([scores[i] for i in indices_validos])
    top_idx_local = scores_filtrados.argsort()[::-1][:top_k]

    resultados = []
    for local in top_idx_local:
        idx_global = indices_validos[local]
        d = docs[idx_global]
        resultados.append({
            "score": float(scores[idx_global]),
            "arquivo": d["arquivo"],
            "modulo": d["modulo"],
            "offset": d["offset"],
            "texto": d["texto"],
        })
    return resultados


In [ ]:
resultados = buscar_trechos(
    "screen capture core pipeline",
    modulo="screenpipe-core",
    top_k=3
)

for r in resultados:
    print("="*80)
    print("Arquivo:", r["arquivo"])
    print("Módulo:", r["modulo"], "| score:", round(r["score"], 3))
    print(r["texto"][:800])


In [ ]:
resultados = buscar_trechos(
    "search api http server sse",
    modulo="screenpipe-server",
    top_k=3
)

for r in resultados:
    print("="*80)
    print("Arquivo:", r["arquivo"])
    print("Módulo:", r["modulo"], "| score:", round(r["score"], 3))
    print(r["texto"][:800])


In [ ]:
resultados = buscar_trechos("core pipeline orchestration", modulo="screenpipe-core", top_k=3)

trecho_combinado = "\n\n".join([r["texto"] for r in resultados])

print(trecho_combinado)

In [ ]:
import json

with open("screenpipe_docs.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)


In [ ]:
import numpy as np

np.save("screenpipe_embeddings.npy", embeddings)


In [ ]:
buscar_trechos("sse stream", modulo="screenpipe-server")


In [ ]:
buscar_trechos("ffmpeg capture", modulo="screenpipe-core")


In [ ]:
buscar_trechos("plugin pipe", modulo="screenpipe-core")


MODELO 2 - DeepSeek Coder


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_p2_id = "deepseek-ai/deepseek-coder-6.7b-instruct"

print("Carregando modelo (DeepSeek Coder):", model_p2_id)

tokenizer_p2 = AutoTokenizer.from_pretrained(model_p2_id)
model_p2 = AutoModelForCausalLM.from_pretrained(
    model_p2_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Dispositivo do modelo:", model_p2.device)


In [ ]:
def deepseek_analisar_texto(texto, question_extra=""):
    # limitar tamanho para não explodir contexto
    texto = texto[:4000]

    prompt = f"""
### Instruction:
Você é um arquiteto de software experiente, com foco em APIs HTTP, SSE (Server-Sent Events) e backends em Rust.

Analise o código/trechos abaixo, que pertencem ao módulo `screenpipe-server` do projeto Screenpipe, e responda:

1. Qual é a responsabilidade principal do `screenpipe-server` dentro do sistema?
2. Quais tipos de APIs ele parece expor (REST, SSE/streaming, etc.)?
3. Como essas APIs se conectam às outras camadas do Screenpipe (core/captura, storage/banco, memória/search)?
4. Que recursos de busca ou filtragem (search) aparecem nesse código?
5. {question_extra}

Responda em português, de forma técnica e estruturada.

### Code:
```rust
{texto}"""
    # 👇 AQUI: só tokenizer_p2
    inputs = tokenizer_p2(prompt, return_tensors="pt").to(model_p2.device)

    with torch.no_grad():
        output_ids = model_p2.generate(
            **inputs,
            max_new_tokens=600,
            pad_token_id=tokenizer_p2.eos_token_id,  # 👈 também tokenizer_p2 aqui
            do_sample=True,
            temperature=0.1,
            top_p=0.95,
        )

    resposta = tokenizer_p2.decode(output_ids[0], skip_special_tokens=True)
    return resposta


In [ ]:



## 3️⃣ Função Pessoa 2: usar BGE → buscar trechos do `screenpipe-server` → mandar pro DeepSeek

def deepseek_analisar_screenpipe_server(query, top_k=4, question_extra=""):
    """
    Pessoa 2:
    1) Usa os embeddings (BGE) para achar trechos relevantes em `screenpipe-server`.
    2) Usa DeepSeek Coder para analisar a arquitetura das APIs / SSE / search.
    """
    resultados = buscar_trechos(query, modulo="screenpipe-server", top_k=top_k)

    # filtrar trechos muito curtos (que só confundem o modelo)
    resultados = [r for r in resultados if len(r["texto"]) > 150]

    if not resultados:
        return "Nenhum trecho relevante ou suficientemente grande encontrado em screenpipe-server."

    partes = []
    for r in resultados:
        header = f"// Arquivo: {r['arquivo']} | Offset: {r['offset']}\n"
        partes.append(header + r["texto"])

    texto_combinado = "\n\n".join(partes)

    return deepseek_analisar_texto(
        texto_combinado,
        question_extra=question_extra
    )


In [ ]:
analise_server_api = deepseek_analisar_screenpipe_server(
    query="http server route sse search api",
    top_k=4,
    question_extra="Destaque claramente quais endpoints parecem ser responsáveis por streaming em tempo real (SSE) e quais lidam com busca/consulta histórica."
)

print(analise_server_api)
